<a href="https://colab.research.google.com/github/Anthei0774/Advent-of-Code/blob/main/2023/Day_5_If_You_Give_A_Seed_A_Fertilizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
puzzle = '''seeds: 79 14 55 13

seed-to-soil map:
50 98 2
52 50 48

soil-to-fertilizer map:
0 15 37
37 52 2
39 0 15

fertilizer-to-water map:
49 53 8
0 11 42
42 0 7
57 7 4

water-to-light map:
88 18 7
18 25 70

light-to-temperature map:
45 77 23
81 45 19
68 64 13

temperature-to-humidity map:
0 69 1
1 0 69

humidity-to-location map:
60 56 37
56 93 4'''

with open('5.txt') as f: puzzle = f.read()

puzzle = puzzle.replace(' map:', '')
puzzle = puzzle.split('\n')
puzzle = [line for line in puzzle if line != '']
puzzle

# preprocessing
seeds = puzzle[0].split(' ')
seeds = seeds[1 :]
seeds = list(map(int, seeds))
seeds

mapping = {}
current = None

i = 1
while i < len(puzzle):
    line = puzzle[i]
    if '-' in line:
        current = line
        mapping[current] = {}
    else:
        line = line.split(' ')
        line = list(map(int, line))
        [destination, source, length] = line
        source = (source, source + length - 1)
        destination = (destination, destination + length - 1)
        mapping[current][source] = destination
    i += 1

mapping

################################################################################
# PART 1

for i, s in enumerate(seeds):
    # print('Seed:', s)

    for m in mapping:
        for source in mapping[m]:
            if source[0] <= s and s <= source[1]:
                s = s - source[0] + mapping[m][source][0]
                break

    seeds[i] = s

min(seeds)

################################################################################
# PART 2

# https://stackoverflow.com/questions/55480499/split-set-of-intervals-into-minimal-set-of-disjoint-intervals
def create_new_intervals(intervals):

    starts = [(i[0], 1) for i in intervals]
    # if verbose: print('Starts:', starts)

    ends = [(i[1] + 1, -1) for i in intervals]
    # if verbose: print('Ends:', ends)

    endpoints = starts + ends
    # endpoints = list(set(endpoints)) # leave commented out, otherwise code is buggy for reasons ... ?
    endpoints = sorted(endpoints, key=lambda i: -i[1])
    endpoints = sorted(endpoints, key=lambda i: i[0])
    # if verbose: print('Endpoints:', endpoints, '\n')

    new_intervals = []
    count = 0
    prev = None
    for i in endpoints:

        new = None
        if prev is not None:
            if prev < i[0] and count != 0:
                new = (prev, i[0] - 1)
        prev = i[0]
        count += i[1]
        new_intervals.append(new)

    new_intervals = [new for new in new_intervals if new is not None]
    new_intervals = list(set(new_intervals))
    new_intervals = sorted(new_intervals, key=lambda i: i[0])

    return new_intervals

# function merges together an X-to-Y and a Y-to-Z mapping into an X-to-Z
# verbose argument since I needed to debug this for a looong time... :/
def merge_mapping(k1, k2, verbose=False):
    global mapping

    # check if mapping can be done
    k1_s = k1.split('-')
    k2_s = k2.split('-')
    assert k1_s[-1] == k2_s[0]

    # take destination intervals from X-to-Y and source intervals from Y-to-Z
    intervals = list(mapping[k1].values()) + list(mapping[k2])
    intervals = sorted(intervals, key=lambda x: x[1])
    intervals = sorted(intervals, key=lambda x: x[0])
    if verbose: print('Intevals:', intervals)

    # *** black magic ***
    new_intervals = create_new_intervals(intervals)
    if verbose: print('New intervals:', new_intervals, '\n')

    # create new mapping
    k = k1_s[0] + '-to-' + k2_s[-1]
    mapping[k] = {}

    # for each new interval... I
    for new in new_intervals:
        if verbose: print('\nNew:', new)

        # try to find an entry like X'-to-Y' where I is part of Y' (X' is then source)
        src_found = False
        for srcsrc in mapping[k1]:
            src = mapping[k1][srcsrc]
            if src[0] <= new[0] and new[1] <= src[1]:
                src_found = True
                break

        # try to find an entry like Y'-to-Z' where I is part of Y' (Z' is then destination)
        dst_found = False
        for dst in mapping[k2]:
            if dst[0] <= new[0] and new[1] <= dst[1]:
                dstdst = mapping[k2][dst]
                dst_found = True
                break

        # either source or destination or both has to be found!
        assert src_found or dst_found

        # if source X' found, shrink to appropriate size
        src_mod = None
        if src_found:
            if verbose: print('Source found:', srcsrc, 'mapped to', src)

            left = new[0] - src[0]
            right = src[1] - new[1]
            assert 0 <= left and 0 <= right

            src_mod = (srcsrc[0] + left, srcsrc[1] - right)
            assert src_mod[1] - src_mod[0] == new[1] - new[0]
        else:
            src_mod = new

        # same with destination Z'
        dst_mod = None
        if dst_found:
            if verbose: print('Destination found:', dst, 'mapped to', dstdst)

            left = new[0] - dst[0]
            right = dst[1] - new[1]
            assert 0 <= left and 0 <= right

            dst_mod = (dstdst[0] + left, dstdst[1] - right)
            assert dst_mod[1] - dst_mod[0] == new[1] - new[0]
        else:
            dst_mod = new

        # final mapping created X'-to-Z' for the new interval I
        if verbose: print('Mapping:', src_mod, dst_mod)
        mapping[k][src_mod] = dst_mod

    del mapping[k1], mapping[k2]
    intervals = sorted(mapping[k], key=lambda x: x[1])
    intervals = sorted(intervals, key=lambda x: x[0])
    mapping[k] = {m : mapping[k][m] for m in intervals}


# run big chungus above, tie together all mappings
verbose = False
merge_mapping('seed-to-soil', 'soil-to-fertilizer', verbose)
merge_mapping('seed-to-fertilizer', 'fertilizer-to-water', verbose)
merge_mapping('seed-to-water', 'water-to-light', verbose)
merge_mapping('seed-to-light', 'light-to-temperature', verbose)
merge_mapping('seed-to-temperature', 'temperature-to-humidity', verbose)
merge_mapping('seed-to-humidity', 'humidity-to-location', verbose)
verbose = True

mapping = mapping['seed-to-location']
mapping

###############################################################################
# check part 1 is still working

# reset seeds first
seeds = puzzle[0].split(' ')
seeds = seeds[1 :]
seeds = list(map(int, seeds))

for i, s in enumerate(seeds):
    for m in mapping:
        if m[0] <= s and s <= m[1]:
            s = mapping[m][0] + (s - m[0])
            break
    seeds[i] = s

min(seeds) # stuff still works :)

###############################################################################
# back to part 2

# reset seeds again
seeds = puzzle[0].split(' ')
seeds = seeds[1 :]
seeds = list(map(int, seeds))
seeds = [(seeds[i], seeds[i] + seeds[i + 1] - 1) for i in range(0, len(seeds), 2)] # transform to ranges
seeds

# zip together seed-ranges with mapping ranges and apply new interval creation ==> result is list of disjoint intervals from where we can filter out the seed-ranges which are exactly mapped
new_seeds = seeds + list(mapping)
new_seeds = sorted(new_seeds, key=lambda i: i[1])
new_seeds = sorted(new_seeds, key=lambda i: i[0])
new_seeds = create_new_intervals(new_seeds)
new_seeds = [n_s for n_s in new_seeds for s in seeds if s[0] <= n_s[0] and n_s[1] <= s[1]]
new_seeds = [n_s[0] for n_s in new_seeds] # take only the beginning of each interval, since mapping is like x + b for all entries with b varying
new_seeds

for i, s in enumerate(new_seeds):
    for m in mapping:
        if m[0] <= s and s <= m[1]:
            s = mapping[m][0] + (s - m[0])
            break
    new_seeds[i] = s

min(new_seeds)